## Configuration

In [ ]:
from pathlib import Path
import tomllib
from utils.utils import *
import json

PATH_EEG = Path("../EEG/EEG/results/")
FILENAME = 'Case_04_ExG.csv'
MARKER_FILE = 'Case_04_Marker.csv'
CONFIG_PATH = "../EEG/EEG/config/config.toml"

with open(CONFIG_PATH, "rb") as f:
    config = tomllib.load(f)

BANDS = config['BANDS']
SEGMENT_DEF = config['SEGMENT_DEF']
RANDOM_SEED = config['RANDOM_SEED']
DURATION = config['DURATION']

SFREQ = config['SFREQ']
L_FREQ = config['L_FREQ']
H_FREQ = config['H_FREQ']

## Raw Data

In [ ]:
raw_full, df_full, df_markers_full = load_and_preprocess_eeg(PATH_EEG, FILENAME, MARKER_FILE, SFREQ, L_FREQ, H_FREQ)

raws, labels = extract_segments(raw_full, df_full, df_markers_full, SEGMENT_DEF)

In [ ]:
plot_psd_comparison(raws, labels)

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Alpha')
plot_topomap_comparison(raws, labels, band_name='Beta')

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Delta')
plot_topomap_comparison(raws, labels, band_name='Theta')

## Preprocessing

In [ ]:
import mne
from mne.preprocessing import ICA
raw, df, df_markers = load_and_preprocess_eeg(PATH_EEG, FILENAME, MARKER_FILE, SFREQ, L_FREQ, H_FREQ)

# ICA
# n_components=15 (16 Channels, ICA finding max n_channels - 1 or same number)
ica = ICA(n_components=15, max_iter='auto', random_state=RANDOM_SEED)

print(">>> ICA (Might take a while)...")
ica.fit(raw)

print(">>> Showing Components...")
ica.plot_components()
plt.show()

ica.plot_sources(raw, show_scrollbars=True)
plt.show()

In [ ]:
exclude_indices = [0, 1, 3, 7, 10] 

print(f">>> Removing ingredients: {exclude_indices}")
ica.exclude = exclude_indices

raw_clean = raw.copy()
ica.apply(raw_clean)

print(">>> Cleaned data.")

target_chan = 'F3' if 'F3' in raw.ch_names else raw.ch_names[0]

print(f"Comparison on the channel {target_chan}...")
fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(raw.get_data(picks=target_chan)[0, 1000:2500], color='red', alpha=0.5, label='Before cleaning (Original)')
ax.plot(raw_clean.get_data(picks=target_chan)[0, 1000:2500], color='black', label='After cleaning (Clean)')
ax.set_title(f"Artifact Removal Effect (Eyes and Muscles) - Channel {target_chan}")
ax.legend()
plt.show()

In [ ]:
raws, labels = extract_segments(raw_clean, df_full, df_markers_full, SEGMENT_DEF)

plot_psd_comparison(raws, labels)

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Alpha')
plot_topomap_comparison(raws, labels, band_name='Beta')

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Delta')
plot_topomap_comparison(raws, labels, band_name='Gamma')

### Analiza Pasma Alpha (8–12 Hz): Asymetria i Sterowanie
* **Dominacja Ręki:** Wyraźna dysocjacja C3 (Niebieskie - Praca) vs C4 (Czerwone - Spoczynek). Mózg precyzyjnie izoluje prawą rękę do obsługi telefonu, aktywnie hamując lewą stronę ciała.
* **Faza 1 vs Faza 4:**
    * **Faza 1 (Brainrot):** Wysoka Alpha z tyłu głowy (>6 w Delcie, Czerwień w Alpha). Mózg wchodzi w stan "pustego patrzenia".
    * **Faza 4 (Smart):** Alpha znika (robi się niebieska) w obszarach kojarzeniowych. To jedyny moment, gdy mózg faktycznie "wychodzi z trybu uśpienia".

### Analiza Pasma Beta (13–30 Hz): "Efekt Tunelowy"
* **Obserwacja:** Cały mózg pozostaje wyciszony (Niebieski), z wyjątkiem lewej kory wzrokowej (O1), która jest stale aktywna (Czerwona).
* **Wniosek:** Uczestnik filtruje bodźce. Nie "chłonie" atmosfery wideo (muzyka, kolory), lecz skupia się na wyciąganiu konkretnych informacji (prawdopodobnie czytanie napisów/analiza detali). Najsilniejsza aktywacja w **Fazie 4** potwierdza, że treści Smart wymagają najintensywniejszej pracy analitycznej.

### Analiza Pasma Delta (1–4 Hz): Poziomy Świadomości
* **Hierarchia Wybudzenia:**
    1.  **Faza 1** Delta > 6 (Stan bliski snu/transu). Najniższa świadomość.
    2.  **Faza 3** Delta ~5. Ruch ręką lekko wybudza mózg.
    3.  **Faza 4** Delta ~4. Najwyższy poziom przytomności wymuszony "trudnością" treści.
* **Wniosek:** Brainrot bez interakcji (scrollowania) powoduje niemal całkowite sensoryczne odcięcie kory wzrokowej.

### Analiza Pasma Gamma (>30 Hz): Koszt Fizjologiczny
* **Scrollowanie (Fazy 3 i 4):** Gwałtowny wzrost Gammy (>1.0) w O1.
* **Interpretacja:** Wskażnik pracy fizycznej oczu oraz scalania tekstu.
* **Dominacja Fazy 4:** Największy obszar aktywności ("rozlanie" za O1) w Fazie Smart sugeruje, że czytanie ze zrozumieniem i łączenie faktów obciąża lewą półkulę znacznie bardziej niż mechaniczne scrollowanie brainrotu.

---

1.  **Odporność na "Chaos":** Uczestnik nie ulega przebodźcowaniu brainrotem (brak "pożaru" w całej głowie). Zamiast tego, jego mózg wchodzi w tryb oszczędzania energii (wysoka Delta), ignorując większość bodźców.
2.  **Smart Content jako Aktywator:** Jedynie połączenie **Ruchu (Scroll)** i **Wymagającej Treści (Smart)** jest w stanie przełamać barierę ochronną tego mózgu i zmusić go do pełnej aktywności (zniknięcie Alphy, spadek Delty).
3.  **Styl Przetwarzania:** Wybitnie tekstowy/zadaniowy. Uczestnik "czyta internet", zamiast go oglądać.

## Spektogram

In [ ]:
# Alpha (fatigue/relax)
plot_band_power_over_time(raws, labels, band=(8, 12), band_name='Alpha')
# Beta (Focus)
plot_band_power_over_time(raws, labels, band=(13, 30), band_name='Beta')

In [ ]:
# Emotion
calculate_asymmetry(raws, labels, ch_left='F3', ch_right='F4')

# According to Heller's model, parietal asymmetry is responsible for physiological arousal
calculate_asymmetry(raws, labels, ch_left='P3', ch_right='P4')

# Image processing method.
calculate_asymmetry(raws, labels, ch_left='O1', ch_right='O2')

# Motor
calculate_asymmetry(raws, labels, ch_left='C3', ch_right='C4')

### Podsumowanie

**F4/F3**: Wyniki rzędu 2.0 w asymetrii czołowej sugerują bardzo silny system nagrody. Scrollowanie sprawia przyjemność i daje poczucie kontroli.

**P4/P3**: Ciało nie reaguje stresem (niskie pobudzenie prawej półkuli). Jest w strefie komfortu.

**Faza O2/O1**: Wzrokowiec. Nawet przy treściach Smart (Smart Scroll) prawa półkula wciąż lekko dominuje, co sugeruje, że nawet wykresy/tekst przetwarza jako obrazy.

## Scroll

In [ ]:
PROJECT_ROOT = Path.cwd().parent 

USER_MAP_PATH = PROJECT_ROOT / "EEG" / "EEG" / "user_map.json"
INTERACTIONS_PATH = PROJECT_ROOT / "EEG" / "research_events_rows.csv"
USERS_PATH = PROJECT_ROOT / "EEG" / "Users_rows.csv"

FILE_NAME = 'Case_04_ExG.csv'

In [ ]:
scroll_data = extract_scroll_events(FILE_NAME, INTERACTIONS_PATH, USERS_PATH, USER_MAP_PATH)

if scroll_data is not None and not scroll_data.empty:
    new_annotations = mne.Annotations(
        onset=scroll_data['onset'].values,
        duration=scroll_data['duration'].values,
        description=scroll_data['description'].values
    )

    raw_clean.set_annotations(new_annotations)
    
    print("SUCCESS: Annotations applied to raw_clean.")
    print(f"Time range of scrolls: {scroll_data['onset'].min():.2f}s to {scroll_data['onset'].max():.2f}s")
else:
    print("WARNING: No scroll data found. Plots will be empty.")

In [ ]:
compare_stages(
    raw_clean, 
    band='Beta', 
    fmin=13, 
    fmax=30, 
    channel='C3',  
    duration=DURATION
)

In [ ]:
# REWARD/MOTIVATION SYSTEM (Frontal Alpha)
# Channel: F3 (Left Frontal)
# Band: Alpha (8-12 Hz)
# Interpretation: THE LOWER THE GRAPH, THE GREATER THE “REWARD/ENGAGEMENT.”

compare_stages(
    raw_clean, 
    band='Alpha', 
    fmin=8, 
    fmax=12, 
    channel='F3',   # Key channel for positive/aspirational emotions
    duration=DURATION
)

In [ ]:
# ISUAL RELAXATION (Alpha)
# Channel: O1 (Back of the head)
# Band: Alpha (8-12 Hz)
# Interpretation: HIGH LINE = RELAXATION / ZOMBIE MODE. LOW = WORK.

compare_stages(
    raw_clean, 
    band='Alpha', 
    fmin=8, 
    fmax=12, 
    channel='O1', 
    duration=DURATION
)

In [ ]:
# INTELLECTUAL EFFORT (Frontal Theta) ---
# Channel: Fz (Center of forehead)
# Band: Theta (4-8 Hz)
# Interpretation: PEAKS = STRONG THINKING/MEMORIZATION.

target_channel = 'Fz'

compare_stages(
    raw_clean, 
    band='Theta', 
    fmin=4, 
    fmax=8, 
    channel=target_channel, 
    duration=DURATION
)

In [ ]:
# COGNITIVE CONTROL & INHIBITION (Right Frontal Theta) ---
# Channel: F4 (Right forehead / Right Frontal)
# Band: Theta (4-8 Hz)
# Interpretation: PEAKS = SUSTAINED FOCUS & SUPPRESSING DISTRACTIONS (Mental effort to stay on task or regulate emotional impulses).

target_channel = 'F4'

compare_stages(
    raw_clean, 
    band='Theta', 
    fmin=4, 
    fmax=8, 
    channel=target_channel, 
    duration=DURATION
)

In [ ]:
# VERBAL & ANALYTICAL MEMORY (Left Parietal Theta) ---
# Channel: P3 (Left back of head / Left Parietal)
# Band: Theta (4-8 Hz)
# Interpretation: PEAKS = PROCESSING TEXT/LANGUAGE (Reading captions, understanding speech, verbal working memory).

target_channel = 'P3'

compare_stages(
    raw_clean, 
    band='Theta', 
    fmin=4, 
    fmax=8, 
    channel=target_channel, 
    duration=DURATION
)

In [ ]:
# VISUOSPATIAL PROCESSING & ATTENTION (Right Parietal Theta) ---
# Channel: P4 (Right back of head / Right Parietal)
# Band: Theta (4-8 Hz)
# Interpretation: PEAKS = ENCODING VISUAL/SPATIAL INFO (Processing layout, movement, spatial memory, or complex imagery).

target_channel = 'P4'

compare_stages(
    raw_clean, 
    band='Theta', 
    fmin=4, 
    fmax=8, 
    channel=target_channel, 
    duration=DURATION
)

#### Kora ruchowa (Beta C3) – Mechanika ruchu
**Smart Scroll**: Bardzo dobrze widać fizjologiczną reakcję: tuż przed i w trakcie linii sygnał Beta lekko spada (desynchronizacja - przygotowanie ruchu), a po chwili widać szarpnięcie w górę (synchronizacja - zakończenie ruchu).

**Brainrot Scroll**: Palec pracuje tutaj niemal bez przerwy (mnóstwo czerwonych przerywanych linii). Kora ruchowa uczestnika nie ma czasu na "odpoczynek". Linia fali Beta to jeden wielki, ściśnięty szum ciągłej gotowości motorycznej.

#### Kora potyliczna (Alpha O1) – "Wzrokowiec" w akcji
Z wykresów asymetrii wiemy, że ta osoba dosłownie "pożera" świat wizualnie. Wykresy czasowe idealnie to potwierdzają.

**Smart Scroll**: Uczestnik przewija ekran, pojawia się nowa treść, a fioletowa linia fal Alpha natychmiast, drastycznie nurkuje w dół. Kora wzrokowa włącza się, żeby zdekodować nowy obraz (przy fali Alpha spadek = aktywność).

**Brainrot Scroll**: Przez ciągłe miganie nowych nagrań/obrazów, kora wzrokowa nie potrafi ustabilizować sygnału. To ciągłe uderzanie bodźców wizualnych, które badany bardzo chętnie przyjmuje.

#### Fale czołowe i ciemieniowe (Theta Fz, F4, P3, P4) – Zrozumienie

**Smart Scroll**: Widziać zdrowy schemat. Na wykresie Fz | SMART SCROLL. Jest scroll (niebieska linia), ułamek sekundy przerwy, a potem ukształtowana "górka" fali Theta. „Przewięcie -> Zobaczenie -> Zrozumienie”.

**Brainrot Scroll**: Na wykresach Fz, F4 i P4 dla trybu Brainrot oojawia się gigantyczny pik fali Theta. Co ważne – nie ma tam linii scrollowania!
To może być dowód na tzw. "przebodźcowanie" niezależne od motoryki. W trybie Brainrot uczestnik przewija bezmyślnie, ale na ekranie musiało pojawić się coś (filmik, obraz), co wywołało u niego masywny szok poznawczy, potężny strzał dopaminy lub nagłe skupienie uwagi. Jego płaty czołowe i ciemieniowe "eksplodowały" aktywnością z opóźnieniem w stosunku do samego ruchu palcem.

---

Dla badanego Smart Scroll to świadome, logiczne przyswajanie informacji (akcja -> spadek Alpha -> skok Theta).
Z kolei Brainrot Scroll to "dopaminowa zabawa". Sam ruch palcem jest tu tylko mechanicznym nawykiem, który ma utrzymać korę wzrokową w stanie ciągłej ekstazy, a płaty czołowe są losowo bombardowane potężnymi ładunkami emocjonalno-poznawczymi.